# Divvy Bikeshare Lakehouse — 03 · Gold (Facts)

**Run order:** `01_bronze_extract_load` → `02_gold_dimensions` → **`03_gold_facts`**

Builds the fact tables at the centre of the star schema, using the surrogate keys
generated in notebook 02:

| Fact | Grain | Measures |
|---|---|---|
| `gold.fact_trip` | one row per trip | `duration_seconds`, `duration_minutes`, `rider_age_at_trip` |
| `gold.fact_payment` | one row per payment | `amount`, `rider_age_at_payment` |
| `gold.fact_rider_monthly` | one row per rider per active month | rides, ride minutes, amount paid *(extra credit)* |
| `gold.agg_rider_spend_vs_rides` | one row per rider | spend vs ride frequency *(extra credit)* |

`dim_rider` and `dim_date` are **conformed** — both transaction facts key to them, which
is what allows spend and ride behaviour to be compared for the same rider on the same
calendar.

In [0]:
BRONZE_DB = "bronze"
GOLD_DIR = "dbfs:/delta/gold"
GOLD_DB = "gold"

from pyspark.sql import functions as F

spark.conf.set("spark.sql.shuffle.partitions", 8)

# See notebook 01: Unity Catalog will not create tables over `dbfs:` locations, so pin
# the session to the Hive metastore where the bronze and gold Delta files live.
try:
    spark.sql("USE CATALOG hive_metastore")
    print("catalog: hive_metastore")
except Exception:
    print("catalog: workspace default (no Unity Catalog here)")


def write_gold(df, table_name):
    """Write `df` to Delta in overwrite mode and register it as gold.<table_name>.

    Same writer as notebook 02: `saveAsTable` first, falling back to writing the Delta
    files and registering the table over them on runtimes whose V2 catalog refuses an
    overwrite. Either path produces a Delta table, overwritten, in the gold database.
    """
    path = GOLD_DIR + "/" + table_name
    table = GOLD_DB + "." + table_name

    def delta_writer():
        return (df.write
                  .format("delta")
                  .mode("overwrite")
                  .option("overwriteSchema", "true"))

    try:
        delta_writer().option("path", path).saveAsTable(table)
    except Exception as exc:
        print("  saveAsTable unavailable on this runtime (" + type(exc).__name__ +
              "); writing Delta files and registering the table instead")
        delta_writer().save(path)
        spark.sql("DROP TABLE IF EXISTS " + table)
        spark.sql("CREATE TABLE " + table + " USING DELTA LOCATION '" + path + "'")

    n = spark.table(table).count()
    print("wrote", table.ljust(28), format(n, ",").rjust(10), "rows ->", path)
    return n


def age_band(age_col):
    """Same banding as dim_rider, so '30-39' means the same thing everywhere."""
    return (
        F.when(age_col.isNull(), F.lit("unknown"))
         .when(age_col < 20, F.lit("under 20"))
         .when(age_col < 30, F.lit("20-29"))
         .when(age_col < 40, F.lit("30-39"))
         .when(age_col < 50, F.lit("40-49"))
         .when(age_col < 65, F.lit("50-64"))
         .otherwise(F.lit("65+"))
    )

catalog: hive_metastore


## 1 · `fact_trip`

Grain: **one row per trip**.

* Foreign keys to `dim_rider`, `dim_station` (twice — start and end), `dim_date` (twice)
  and `dim_time` (twice).
* `trip_id` and `rideable_type` ride along as degenerate dimensions.
* `duration_minutes` is the additive measure business outcome 1 is built on.
* `rider_age_at_trip` is computed **against the trip date**, not the account start date,
  so a rider who has been on the platform for years is counted at the age they actually
  were on the day of the ride. That is a point-in-time value, which is why it is a fact
  and not a rider attribute — storing it on `dim_rider` would be a slowly-changing
  dimension problem this project does not need.
* `is_member` / `rider_type` are carried onto the fact so outcome 1d needs no join.

Lookups are **left joins**, so no trip is ever silently dropped; anything that fails to
match lands on the `-1` Unknown member and is counted in the audit section below.

In [0]:
trip_src = (
    spark.table(BRONZE_DB + ".trip")
    .withColumn("start_station_id", F.trim(F.col("start_station_id")))
    .withColumn("end_station_id", F.trim(F.col("end_station_id")))
)

rider_lkp = (
    spark.table(GOLD_DB + ".dim_rider")
    .select("rider_key", "rider_id", "birthday", "is_member", "rider_type")
)

start_station_lkp = (
    spark.table(GOLD_DB + ".dim_station")
    .select(F.col("station_key").alias("start_station_key"),
            F.col("station_id").alias("start_station_id"))
)

end_station_lkp = (
    spark.table(GOLD_DB + ".dim_station")
    .select(F.col("station_key").alias("end_station_key"),
            F.col("station_id").alias("end_station_id"))
)

In [0]:
fact_trip = (
    trip_src
    .join(rider_lkp, on="rider_id", how="left")
    .join(start_station_lkp, on="start_station_id", how="left")
    .join(end_station_lkp, on="end_station_id", how="left")

    # --- dimension keys -------------------------------------------------
    .withColumn("rider_key", F.coalesce(F.col("rider_key"), F.lit(-1)))
    .withColumn("start_station_key", F.coalesce(F.col("start_station_key"), F.lit(-1)))
    .withColumn("end_station_key", F.coalesce(F.col("end_station_key"), F.lit(-1)))
    .withColumn("start_date_key", F.date_format("started_at", "yyyyMMdd").cast("int"))
    .withColumn("end_date_key", F.date_format("ended_at", "yyyyMMdd").cast("int"))
    .withColumn("start_time_key", F.hour("started_at").cast("int"))
    .withColumn("end_time_key", F.hour("ended_at").cast("int"))

    # --- measures -------------------------------------------------------
    .withColumn(
        "duration_seconds",
        (F.col("ended_at").cast("long") - F.col("started_at").cast("long")).cast("int"),
    )
    .withColumn(
        "duration_minutes",
        F.round(F.col("duration_seconds") / F.lit(60.0), 2).cast("decimal(10,2)"),
    )
    .withColumn(
        "rider_age_at_trip",
        F.floor(F.months_between(F.to_date("started_at"), F.col("birthday")) / F.lit(12)).cast("int"),
    )
    .withColumn("rider_age_band_at_trip", age_band(F.col("rider_age_at_trip")))

    .select(
        "trip_id",
        "rider_key",
        "start_station_key",
        "end_station_key",
        "start_date_key",
        "end_date_key",
        "start_time_key",
        "end_time_key",
        "rideable_type",
        "started_at",
        "ended_at",
        "duration_seconds",
        "duration_minutes",
        "rider_age_at_trip",
        "rider_age_band_at_trip",
        "is_member",
        "rider_type",
    )
)

write_gold(fact_trip, "fact_trip")

wrote gold.fact_trip                4,584,921 rows -> dbfs:/delta/gold/fact_trip


4584921

In [0]:
%sql
SELECT * FROM gold.fact_trip LIMIT 10

trip_id,rider_key,start_station_key,end_station_key,start_date_key,end_date_key,start_time_key,end_time_key,rideable_type,started_at,ended_at,duration_seconds,duration_minutes,rider_age_at_trip,rider_age_band_at_trip,is_member,rider_type
89E7AA6C29227EFF,70935,454,529,20210212,20210212,16,16,classic_bike,2021-02-12T16:14:56Z,2021-02-12T16:21:43Z,407,6.78,37,30-39,true,Member
0FEFDE2603568365,46855,454,192,20210214,20210214,17,18,classic_bike,2021-02-14T17:52:38Z,2021-02-14T18:12:09Z,1171,19.52,38,30-39,true,Member
E6159D746B2DBB91,69871,552,693,20210209,20210209,19,19,electric_bike,2021-02-09T19:10:18Z,2021-02-09T19:19:10Z,532,8.87,33,30-39,true,Member
B32D3199F1C2E75B,57975,512,696,20210202,20210202,17,17,classic_bike,2021-02-02T17:49:41Z,2021-02-02T17:54:06Z,265,4.42,19,under 20,false,Casual
83E463F23575F4BF,38609,75,827,20210223,20210223,15,15,electric_bike,2021-02-23T15:07:23Z,2021-02-23T15:22:37Z,914,15.23,71,65+,true,Member
BDAA7E3494E8D545,35268,216,656,20210224,20210224,15,15,electric_bike,2021-02-24T15:43:33Z,2021-02-24T15:49:05Z,332,5.53,27,20-29,true,Member
A772742351171257,49105,656,656,20210201,20210201,17,17,classic_bike,2021-02-01T17:47:42Z,2021-02-01T17:48:33Z,51,0.85,33,30-39,true,Member
295476889D9B79F8,18619,216,216,20210211,20210211,18,18,classic_bike,2021-02-11T18:33:53Z,2021-02-11T18:35:09Z,76,1.27,23,20-29,true,Member
362087194BA4CC9A,15733,656,656,20210227,20210227,15,15,classic_bike,2021-02-27T15:13:39Z,2021-02-27T15:36:36Z,1377,22.95,19,under 20,false,Casual
21630F715038CCB0,56069,656,656,20210220,20210220,8,9,classic_bike,2021-02-20T08:59:42Z,2021-02-20T09:17:04Z,1042,17.37,47,40-49,false,Casual


## 2 · `fact_payment`

Grain: **one row per payment**. `amount` is the additive measure. The rider's age at
*account start* lives on `dim_rider` because it is fixed per rider; age at *payment* is
point-in-time and so sits here on the fact, exactly as `rider_age_at_trip` does.

In [0]:
payment_src = spark.table(BRONZE_DB + ".payment")

payment_rider_lkp = (
    spark.table(GOLD_DB + ".dim_rider").select("rider_key", "rider_id", "birthday")
)

fact_payment = (
    payment_src
    .join(payment_rider_lkp, on="rider_id", how="left")
    .withColumn("rider_key", F.coalesce(F.col("rider_key"), F.lit(-1)))
    .withColumn("date_key", F.date_format("date", "yyyyMMdd").cast("int"))
    .withColumn(
        "rider_age_at_payment",
        F.floor(F.months_between(F.col("date"), F.col("birthday")) / F.lit(12)).cast("int"),
    )
    .withColumnRenamed("date", "payment_date")
    .select("payment_id", "rider_key", "date_key", "payment_date", "amount",
            "rider_age_at_payment")
)

write_gold(fact_payment, "fact_payment")

wrote gold.fact_payment             1,946,607 rows -> dbfs:/delta/gold/fact_payment


1946607

In [0]:
%sql
SELECT * FROM gold.fact_payment LIMIT 10

payment_id,rider_key,date_key,payment_date,amount,rider_age_at_payment
1,1,20190501,2019-05-01,9.00,30
2,1,20190601,2019-06-01,9.00,30
3,1,20190701,2019-07-01,9.00,30
4,1,20190801,2019-08-01,9.00,30
5,1,20190901,2019-09-01,9.00,30
6,1,20191001,2019-10-01,9.00,30
7,1,20191101,2019-11-01,9.00,30
8,1,20191201,2019-12-01,9.00,30
9,1,20200101,2020-01-01,9.00,30
10,1,20200201,2020-02-01,9.00,30


## 3 · `fact_rider_monthly` — extra credit

Grain: **one row per rider per calendar month in which the rider was active.**

The third business outcome compares money spent against ride *behaviour* — rides per
month and minutes per month. Those live in two different facts at two different grains,
so they are conformed here onto a common monthly grain rather than joined at query
time. Joining `fact_trip` to `fact_payment` directly would fan out — every trip against
every payment for that rider — and silently multiply both measures. Aggregating each
side first and then outer-joining keeps the arithmetic honest.

The outer join is deliberate: a rider can pay in a month with no rides, and ride in a
month with no payment. Either would be dropped by an inner join.

In [0]:
date_lkp = spark.table(GOLD_DB + ".dim_date").select("date_key", "month_start_date")

trip_month = (
    spark.table(GOLD_DB + ".fact_trip")
    .join(date_lkp, F.col("start_date_key") == F.col("date_key"), "left")
    .groupBy("rider_key", "month_start_date")
    .agg(
        F.count(F.lit(1)).cast("int").alias("rides_in_month"),
        F.sum("duration_minutes").cast("decimal(12,2)").alias("ride_minutes_in_month"),
    )
)

payment_month = (
    spark.table(GOLD_DB + ".fact_payment")
    .join(date_lkp, on="date_key", how="left")
    .groupBy("rider_key", "month_start_date")
    .agg(
        F.count(F.lit(1)).cast("int").alias("payments_in_month"),
        F.sum("amount").cast("decimal(12,2)").alias("amount_paid_in_month"),
    )
)

fact_rider_monthly = (
    trip_month
    .join(payment_month, on=["rider_key", "month_start_date"], how="full_outer")
    .withColumn("rides_in_month", F.coalesce(F.col("rides_in_month"), F.lit(0)))
    .withColumn("ride_minutes_in_month",
                F.coalesce(F.col("ride_minutes_in_month"), F.lit(0).cast("decimal(12,2)")))
    .withColumn("payments_in_month", F.coalesce(F.col("payments_in_month"), F.lit(0)))
    .withColumn("amount_paid_in_month",
                F.coalesce(F.col("amount_paid_in_month"), F.lit(0).cast("decimal(12,2)")))
    .withColumn("month_date_key", F.date_format("month_start_date", "yyyyMMdd").cast("int"))
    .withColumn("year_month", F.date_format("month_start_date", "yyyy-MM"))
    .withColumn("year", F.year("month_start_date"))
    .withColumn("quarter", F.quarter("month_start_date"))
    .withColumn("month", F.month("month_start_date"))
    .select("rider_key", "month_date_key", "month_start_date", "year_month",
            "year", "quarter", "month", "rides_in_month", "ride_minutes_in_month",
            "payments_in_month", "amount_paid_in_month")
)

write_gold(fact_rider_monthly, "fact_rider_monthly")

wrote gold.fact_rider_monthly       2,049,370 rows -> dbfs:/delta/gold/fact_rider_monthly


2049370

In [0]:
%sql
SELECT * FROM gold.fact_rider_monthly ORDER BY rider_key, month_date_key LIMIT 12

rider_key,month_date_key,month_start_date,year_month,year,quarter,month,rides_in_month,ride_minutes_in_month,payments_in_month,amount_paid_in_month
1,20190501,2019-05-01,2019-05,2019,2,5,0,0.00,1,9.00
1,20190601,2019-06-01,2019-06,2019,2,6,0,0.00,1,9.00
1,20190701,2019-07-01,2019-07,2019,3,7,0,0.00,1,9.00
1,20190801,2019-08-01,2019-08,2019,3,8,0,0.00,1,9.00
1,20190901,2019-09-01,2019-09,2019,3,9,0,0.00,1,9.00
1,20191001,2019-10-01,2019-10,2019,4,10,0,0.00,1,9.00
1,20191101,2019-11-01,2019-11,2019,4,11,0,0.00,1,9.00
1,20191201,2019-12-01,2019-12,2019,4,12,0,0.00,1,9.00
1,20200101,2020-01-01,2020-01,2020,1,1,0,0.00,1,9.00
1,20200201,2020-02-01,2020-02,2020,1,2,0,0.00,1,9.00


## 4 · `agg_rider_spend_vs_rides` — extra credit

Grain: **one row per rider.** Rolls the monthly fact up to lifetime totals and derives
the ratios the business outcome actually asks for.

`months_observed` — first to last active month inclusive — is the denominator rather
than a raw count of rows, so a rider who took a three-month break is not flattered by
having their spend divided only by the months they showed up. `avg_spend_per_month` is
then tenure-neutral: a rider who paid for twelve months and one who paid for two are
compared on the same footing.

In [0]:
monthly = spark.table(GOLD_DB + ".fact_rider_monthly")

rider_attrs = (
    spark.table(GOLD_DB + ".dim_rider")
    .select("rider_key", "rider_type", "is_member",
            "age_at_account_start", "age_band_at_account_start")
)

agg_rider_spend_vs_rides = (
    monthly
    .groupBy("rider_key")
    .agg(
        F.min("month_start_date").alias("first_active_month"),
        F.max("month_start_date").alias("last_active_month"),
        F.sum(F.when(F.col("rides_in_month") > 0, 1).otherwise(0)).cast("int").alias("months_active"),
        F.sum("rides_in_month").cast("int").alias("total_rides"),
        F.sum("ride_minutes_in_month").cast("decimal(12,2)").alias("total_ride_minutes"),
        F.sum("amount_paid_in_month").cast("decimal(12,2)").alias("total_paid"),
    )
    .withColumn(
        "months_observed",
        (F.floor(F.months_between(F.col("last_active_month"), F.col("first_active_month"))) + F.lit(1)).cast("int"),
    )
    .withColumn(
        "avg_rides_per_month",
        F.round(F.col("total_rides") / F.col("months_observed"), 2).cast("decimal(10,2)"),
    )
    .withColumn(
        "avg_spend_per_month",
        F.round(F.col("total_paid") / F.col("months_observed"), 2).cast("decimal(12,2)"),
    )
    .withColumn(
        "spend_per_ride",
        F.when(F.col("total_rides") > 0,
               F.round(F.col("total_paid") / F.col("total_rides"), 2)).cast("decimal(12,2)"),
    )
    .withColumn(
        "rides_per_month_band",
        F.when(F.col("avg_rides_per_month") < 1, F.lit("a. 0-1"))
         .when(F.col("avg_rides_per_month") < 3, F.lit("b. 1-3"))
         .when(F.col("avg_rides_per_month") < 6, F.lit("c. 3-6"))
         .when(F.col("avg_rides_per_month") < 12, F.lit("d. 6-12"))
         .otherwise(F.lit("e. 12+")),
    )
    .join(rider_attrs, on="rider_key", how="left")
    .select("rider_key", "rider_type", "is_member", "age_at_account_start",
            "age_band_at_account_start", "first_active_month", "last_active_month",
            "months_observed", "months_active", "total_rides", "total_ride_minutes",
            "total_paid", "avg_rides_per_month", "avg_spend_per_month",
            "spend_per_ride", "rides_per_month_band")
)

write_gold(agg_rider_spend_vs_rides, "agg_rider_spend_vs_rides")

wrote gold.agg_rider_spend_vs_rides     74,116 rows -> dbfs:/delta/gold/agg_rider_spend_vs_rides


74116

In [0]:
%sql
SELECT * FROM gold.agg_rider_spend_vs_rides ORDER BY rider_key LIMIT 10

rider_key,rider_type,is_member,age_at_account_start,age_band_at_account_start,first_active_month,last_active_month,months_observed,months_active,total_rides,total_ride_minutes,total_paid,avg_rides_per_month,avg_spend_per_month,spend_per_ride,rides_per_month_band
1,Member,true,30,30-39,2019-05-01,2022-02-01,34,8,17,364.87,306.00,0.50,9.00,18.00,a. 0-1
2,Member,true,43,40-49,2019-12-01,2020-09-01,10,0,0,0.00,90.00,0.00,9.00,null,a. 0-1
3,Member,true,23,20-29,2021-07-01,2021-07-01,1,1,1,9.15,0.00,1.00,0.00,0.00,b. 1-3
4,Casual,false,20,20-29,2019-09-01,2022-02-01,30,12,132,2020.42,411.07,4.40,13.70,3.11,c. 3-6
5,Member,true,50,50-64,2019-10-01,2022-02-01,29,0,0,0.00,261.00,0.00,9.00,null,a. 0-1
6,Casual,false,45,40-49,2020-04-01,2022-02-01,23,0,0,0.00,332.59,0.00,14.46,null,a. 0-1
7,Member,true,16,under 20,2020-12-01,2022-01-01,14,11,201,2651.38,117.00,14.36,8.36,0.58,e. 12+
8,Casual,false,28,20-29,2017-01-01,2022-02-01,62,11,203,2683.06,843.59,3.27,13.61,4.16,c. 3-6
9,Member,true,34,30-39,2021-03-01,2022-01-01,11,11,291,4309.18,36.00,26.45,3.27,0.12,e. 12+
10,Member,true,39,30-39,2020-07-01,2021-11-01,17,0,0,0.00,153.00,0.00,9.00,null,a. 0-1


## 5 · Audit the star schema

Referential integrity first — every foreign key on a fact must resolve to a real row in
its dimension — then a data-quality read on the trip measures.

In [0]:
spark.sql("""
    SELECT 'fact_trip.rider_key -> dim_rider' AS relationship, COUNT(*) AS orphan_rows
      FROM gold.fact_trip f LEFT ANTI JOIN gold.dim_rider d ON f.rider_key = d.rider_key
    UNION ALL
    SELECT 'fact_trip.start_station_key -> dim_station', COUNT(*)
      FROM gold.fact_trip f LEFT ANTI JOIN gold.dim_station d ON f.start_station_key = d.station_key
    UNION ALL
    SELECT 'fact_trip.end_station_key -> dim_station', COUNT(*)
      FROM gold.fact_trip f LEFT ANTI JOIN gold.dim_station d ON f.end_station_key = d.station_key
    UNION ALL
    SELECT 'fact_trip.start_date_key -> dim_date', COUNT(*)
      FROM gold.fact_trip f LEFT ANTI JOIN gold.dim_date d ON f.start_date_key = d.date_key
    UNION ALL
    SELECT 'fact_trip.end_date_key -> dim_date', COUNT(*)
      FROM gold.fact_trip f LEFT ANTI JOIN gold.dim_date d ON f.end_date_key = d.date_key
    UNION ALL
    SELECT 'fact_trip.start_time_key -> dim_time', COUNT(*)
      FROM gold.fact_trip f LEFT ANTI JOIN gold.dim_time d ON f.start_time_key = d.time_key
    UNION ALL
    SELECT 'fact_payment.rider_key -> dim_rider', COUNT(*)
      FROM gold.fact_payment f LEFT ANTI JOIN gold.dim_rider d ON f.rider_key = d.rider_key
    UNION ALL
    SELECT 'fact_payment.date_key -> dim_date', COUNT(*)
      FROM gold.fact_payment f LEFT ANTI JOIN gold.dim_date d ON f.date_key = d.date_key
""").show(20, truncate=False)

+------------------------------------------+-----------+
|relationship                              |orphan_rows|
+------------------------------------------+-----------+
|fact_trip.rider_key -> dim_rider          |0          |
|fact_trip.start_station_key -> dim_station|0          |
|fact_trip.end_station_key -> dim_station  |0          |
|fact_trip.start_date_key -> dim_date      |0          |
|fact_trip.end_date_key -> dim_date        |0          |
|fact_trip.start_time_key -> dim_time      |0          |
|fact_payment.rider_key -> dim_rider       |0          |
|fact_payment.date_key -> dim_date         |0          |
+------------------------------------------+-----------+



How many facts landed on the `-1` Unknown member, and how do the trip measures look?
Non-positive durations are **reported, not deleted** — the gold layer stays a complete
account of what the source system said, and an analyst can filter them at query time.

In [0]:
spark.sql("""
    SELECT
        COUNT(*)                                                AS trips,
        SUM(CASE WHEN rider_key         = -1 THEN 1 ELSE 0 END) AS unknown_rider,
        SUM(CASE WHEN start_station_key = -1 THEN 1 ELSE 0 END) AS unknown_start_station,
        SUM(CASE WHEN end_station_key   = -1 THEN 1 ELSE 0 END) AS unknown_end_station,
        SUM(CASE WHEN duration_minutes <= 0 THEN 1 ELSE 0 END)  AS non_positive_duration,
        ROUND(MIN(duration_minutes), 2)                         AS min_minutes,
        ROUND(AVG(duration_minutes), 2)                         AS avg_minutes,
        ROUND(MAX(duration_minutes), 2)                         AS max_minutes,
        MIN(rider_age_at_trip)                                  AS min_age,
        MAX(rider_age_at_trip)                                  AS max_age
    FROM gold.fact_trip
""").show(truncate=False)

+-------+-------------+---------------------+-------------------+---------------------+-----------+-----------+-----------+-------+-------+
|trips  |unknown_rider|unknown_start_station|unknown_end_station|non_positive_duration|min_minutes|avg_minutes|max_minutes|min_age|max_age|
+-------+-------------+---------------------+-------------------+---------------------+-----------+-----------+-----------+-------+-------+
|4584921|0            |0                    |0                  |197                  |-55.90     |21.79      |55944.15   |14     |75     |
+-------+-------------+---------------------+-------------------+---------------------+-----------+-----------+-----------+-------+-------+



In [0]:
spark.sql("SHOW TABLES IN " + GOLD_DB).show(truncate=False)

+--------+------------------------+-----------+
|database|tableName               |isTemporary|
+--------+------------------------+-----------+
|gold    |agg_rider_spend_vs_rides|false      |
|gold    |dim_date                |false      |
|gold    |dim_rider               |false      |
|gold    |dim_station             |false      |
|gold    |dim_time                |false      |
|gold    |fact_payment            |false      |
|gold    |fact_rider_monthly      |false      |
|gold    |fact_trip               |false      |
+--------+------------------------+-----------+



## 6 · The business outcomes, answered

Each query below is one of the questions from the project brief, run against the star
schema exactly as an analyst would run it.

### Outcome 1a — time spent per ride, by day of week and time of day

In [0]:
%sql
SELECT
    d.day_name,
    t.time_of_day,
    COUNT(*)                          AS rides,
    ROUND(AVG(f.duration_minutes), 2) AS avg_minutes
FROM gold.fact_trip f
JOIN gold.dim_date d ON f.start_date_key = d.date_key
JOIN gold.dim_time t ON f.start_time_key = t.time_key
GROUP BY d.day_name, d.day_of_week, t.time_of_day
ORDER BY d.day_of_week, avg_minutes DESC

day_name,time_of_day,rides,avg_minutes
Sunday,Night,108340,29.10
Sunday,Afternoon,299284,28.39
Sunday,Evening,159124,27.32
Sunday,Morning,146778,26.29
Monday,Night,59945,24.73
Monday,Afternoon,186362,22.89
Monday,Evening,190894,19.51
Monday,Morning,138904,17.34
Tuesday,Night,61356,21.47
Tuesday,Afternoon,179030,19.41


### Outcome 1b — time spent per ride, by starting station (top 15 by volume)

In [0]:
%sql
SELECT
    s.station_name                    AS start_station,
    COUNT(*)                          AS rides,
    ROUND(AVG(f.duration_minutes), 2) AS avg_minutes
FROM gold.fact_trip f
JOIN gold.dim_station s ON f.start_station_key = s.station_key
GROUP BY s.station_name
ORDER BY rides DESC
LIMIT 15

start_station,rides,avg_minutes
Streeter Dr & Grand Ave,80344,38.98
Lake Shore Dr & North Blvd,46380,29.18
Lake Shore Dr & Monroe St,44672,39.91
Michigan Ave & Oak St,42722,36.33
Wells St & Concord Ln,41604,15.85
Millennium Park,40505,48.28
Clark St & Elm St,39346,17.44
Wells St & Elm St,35955,13.91
Theater on the Lake,35704,29.27
Kingsbury St & Kinzie St,32422,11.88


### Outcome 1c / 1d — time spent per ride, by rider age at time of trip and membership

In [0]:
%sql
SELECT
    f.rider_age_band_at_trip          AS age_band_at_trip,
    f.rider_type,
    COUNT(*)                          AS rides,
    ROUND(AVG(f.duration_minutes), 2) AS avg_minutes
FROM gold.fact_trip f
GROUP BY f.rider_age_band_at_trip, f.rider_type
ORDER BY age_band_at_trip, f.rider_type

age_band_at_trip,rider_type,rides,avg_minutes
20-29,Casual,305249,21.20
20-29,Member,1236095,22.03
30-39,Casual,279909,21.19
30-39,Member,1073861,22.14
40-49,Casual,145783,21.46
40-49,Member,583115,21.40
50-64,Casual,68121,22.16
50-64,Member,267785,20.96
65+,Casual,7621,22.65
65+,Member,22078,23.15


### Outcome 2a — money spent per month, quarter and year

In [0]:
%sql
SELECT
    d.year,
    d.quarter,
    d.month,
    d.month_name,
    COUNT(*)                AS payments,
    ROUND(SUM(p.amount), 2) AS total_amount
FROM gold.fact_payment p
JOIN gold.dim_date d ON p.date_key = d.date_key
GROUP BY ROLLUP (d.year, d.quarter, d.month, d.month_name)
ORDER BY d.year, d.quarter, d.month

year,quarter,month,month_name,payments,total_amount
null,null,null,null,1946607,19457105.25
2013,null,null,null,5351,53693.34
2013,1,null,null,82,830.65
2013,1,2,February,1,12.90
2013,1,2,null,1,12.90
2013,1,3,null,81,817.75
2013,1,3,March,81,817.75
2013,2,null,null,805,8164.66
2013,2,4,April,169,1672.65
2013,2,4,null,169,1672.65


### Outcome 2b — money spent per rider, by age at account start

In [0]:
%sql
SELECT
    r.age_band_at_account_start,
    r.rider_type,
    COUNT(DISTINCT r.rider_key)                          AS riders,
    ROUND(SUM(p.amount), 2)                              AS total_amount,
    ROUND(SUM(p.amount) / COUNT(DISTINCT r.rider_key), 2) AS amount_per_rider
FROM gold.fact_payment p
JOIN gold.dim_rider r ON p.rider_key = r.rider_key
GROUP BY r.age_band_at_account_start, r.rider_type
ORDER BY r.age_band_at_account_start, r.rider_type

age_band_at_account_start,rider_type,riders,total_amount,amount_per_rider
20-29,Casual,4891,1812717.47,370.62
20-29,Member,19638,4666464.00,237.62
30-39,Casual,3929,1333338.58,339.36
30-39,Member,15899,3513438.00,220.98
40-49,Casual,2007,650274.98,324.00
40-49,Member,8310,1712889.00,206.12
50-64,Casual,815,238537.42,292.68
50-64,Member,3342,639621.00,191.39
65+,Casual,74,22469.48,303.64
65+,Member,273,39375.00,144.23


### Outcome 3 (extra credit) — money spent per rider vs rides and minutes per month

In [0]:
%sql
SELECT
    rides_per_month_band,
    COUNT(*)                                             AS riders,
    ROUND(AVG(avg_rides_per_month), 2)                   AS avg_rides_per_month,
    ROUND(AVG(total_ride_minutes / months_observed), 1)  AS avg_minutes_per_month,
    ROUND(AVG(avg_spend_per_month), 2)                   AS avg_spend_per_month,
    ROUND(AVG(spend_per_ride), 2)                        AS avg_spend_per_ride,
    ROUND(AVG(total_paid), 2)                            AS avg_lifetime_spend
FROM gold.agg_rider_spend_vs_rides
WHERE rider_key <> -1
GROUP BY rides_per_month_band
ORDER BY rides_per_month_band

rides_per_month_band,riders,avg_rides_per_month,avg_minutes_per_month,avg_spend_per_month,avg_spend_per_ride,avg_lifetime_spend
a. 0-1,49291,0.08,2.8,9.59,99.59,282.81
b. 1-3,7344,1.84,41.2,7.92,4.73,318.40
c. 3-6,5892,4.33,91.3,7.65,1.85,255.96
d. 6-12,5647,8.55,175.3,7.12,0.87,180.64
e. 12+,5942,22.89,503.0,5.94,0.32,109.51


### Gold complete

The gold data store holds four dimensions and four facts, every one a Delta table:

```
gold.dim_rider     gold.fact_trip
gold.dim_station   gold.fact_payment
gold.dim_date      gold.fact_rider_monthly         (extra credit)
gold.dim_time      gold.agg_rider_spend_vs_rides   (extra credit)
```